# TR-WISARD — Experimentos em todos os datasets
Roda **TrWisard1**  e **TrWisard2**  com os parâmetros ótimos de cada dataset
e salva métricas, gráficos e vídeo em `data/{dataset}/experimentos/experimento_N/`
(auto-incrementado a cada execução).

In [6]:
import sys, time, json, subprocess, tempfile
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.tr_wisard import TRWisard
from src.dataset import load_dataset, preload_frames
from src.metrics import save_comparison, create_experiment_folder

summary = {}


def run_isolated(mode, dataset, params):
    """Roda um tracker em processo próprio (python src/run_isolated.py) e retorna (predictions, elapsed).

    Necessário para o Tr-WiSARD1 (ClusWisard): instanciar vários ClusWisard na mesma
    sessão Python contamina estado global da extensão C++ `wisardpkg` entre datasets,
    dando métricas erradas a partir do segundo dataset rodado. Rodar cada tracker
    isolado em subprocesso evita o problema.

    `params` é sempre repassado explicitamente ao subprocesso (via arquivo temporário),
    então editar os dicionários PARAMS_TRWISARD1/2 acima continua tendo efeito.
    """
    with tempfile.TemporaryDirectory() as tmp:
        params_path = Path(tmp) / "params.json"
        out_path = Path(tmp) / "result.json"
        with open(params_path, "w") as f:
            json.dump(params, f)
        subprocess.run(
            [sys.executable, str(PROJECT_ROOT / "src" / "run_isolated.py"),
             "--mode", mode, "--dataset", dataset,
             "--params-file", str(params_path), "--out", str(out_path)],
            check=True,
        )
        with open(out_path) as f:
            data = json.load(f)
    return data["predictions"], data["elapsed"]

## Parâmetros por dataset
Parâmetros ótimos tunados, copiados manualmente para os dicionários abaixo — edite aqui
diretamente para testar outros valores (nada é lido de arquivo em tempo de execução).

**TrWisard1**  — valores tunados por `tune_datasets.py` / `tune_faceocc.py` / `tune_tiger1.py`.

**TrWisard2**  — Tabela 4.6.

In [7]:
_CLUS_FIXED = {
    'CLUSWISARD_THRESHOLD':           1,
    'CLUSWISARD_BLEACHING_ACTIVATED': True,
    'CLUSWISARD_ACTIVATION_DEGREE':   True,
    'CLUSWISARD_RETURN_CONFIDENCE':   True,
    'CLUSWISARD_CLASSES_DEGREES':     True,
    'REMOVE_BACKGROUND': False,
    'BACKGROUND_ALPHA':  0,
    'PASS_SCORE':         0.90,
    'SEED':               21,
}

# TrWisard1  — parâmetros tunados por dataset
# Addr | Min.Score | Disc.Limit | Anchor | Scale | Step
PARAMS_TRWISARD1 = {
    'dollar':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 7, 'CLUSWISARD_MIN_SCORE': 0.20, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  5, 'ANCHOR_SCORE': 0.05, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 3},
    'david':    {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 6, 'CLUSWISARD_MIN_SCORE': 0.769, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  6, 'ANCHOR_SCORE': 0.529, 'MAX_SEARCH_WINDOW_SCALE': 0.208, 'STEP_SIZE': 2, 'PASS_SCORE': 0.777},
    'faceocc':  {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 8, 'CLUSWISARD_MIN_SCORE': 0.501, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.637, 'MAX_SEARCH_WINDOW_SCALE': 0.208, 'STEP_SIZE': 5, 'PASS_SCORE': 0.755},
    'faceocc2': {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 6, 'CLUSWISARD_MIN_SCORE': 0.746, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  6, 'ANCHOR_SCORE': 0.54, 'MAX_SEARCH_WINDOW_SCALE': 0.155, 'STEP_SIZE': 2, 'PASS_SCORE': 0.815},
    'sylv':     {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 3, 'CLUSWISARD_MIN_SCORE': 0.77, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  10, 'ANCHOR_SCORE': 0.828, 'MAX_SEARCH_WINDOW_SCALE': 0.153, 'STEP_SIZE': 2, 'PASS_SCORE': 0.803},
    'tiger1':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 6, 'CLUSWISARD_MIN_SCORE': 0.711, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  6, 'ANCHOR_SCORE': 0.758, 'MAX_SEARCH_WINDOW_SCALE': 0.374, 'STEP_SIZE': 1, 'PASS_SCORE': 0.957},
    'tiger2':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 7, 'CLUSWISARD_MIN_SCORE': 0.504, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  9, 'ANCHOR_SCORE': 0.119, 'MAX_SEARCH_WINDOW_SCALE': 0.461, 'STEP_SIZE': 4, 'PASS_SCORE': 0.804},
}

_WISD_FIXED = {'SEED': 21}

# TrWisard2  — parâmetros tunados por dataset
# Addr | Lim.Ret | Lim.ND | Queue | Max.Ret | Radius | Step | BG Rem | α
PARAMS_TRWISARD2 = {
    'dollar':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0.10},
    'david':    {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'faceocc':  {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.20, 'QUEUE_MAX_SIZE':  5, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'faceocc2': {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 5, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.40, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'sylv':     {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS':  5, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 0.10},
    'tiger1':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 10, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS': 30, 'STEP_SIZE': 3, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0},
    'tiger2':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 5, 'LIMIAR_RETREINO': 0.65, 'LIMIAR_NOVO_DISC': 0.40, 'QUEUE_MAX_SIZE':  8, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 20, 'STEP_SIZE': 3, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0},
}

## Execução por dataset
Cada célula abaixo é independente — execute uma de cada vez ou todas em sequência.

In [8]:
# ── dollar ────────────────────────────────────────────────────────────────────
dataset = 'dollar'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'Tr-WiSARD1', elapsed_1,
    preds_2, 'Tr-WiSARD2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== dollar ===
Pré-carregando 327 frames...
  Done em 0.4s
Pasta: experimento_6
Pré-carregando 327 frames...
  Done em 0.3s


TrWisard1: 100%|██████████| 326/326 [00:10<00:00, 32.51it/s]


Pré-carregando 327 frames...
  Done em 0.3s


TrWisard2: 100%|██████████| 326/326 [00:07<00:00, 41.59it/s] 


  Tr-WiSARD1    CLE=4.82 px  Jaccard=84.64%  FPS=32.5
  Tr-WiSARD2    CLE=2.58 px  Jaccard=92.09%  FPS=41.5


In [9]:
# ── david ─────────────────────────────────────────────────────────────────────
dataset = 'david'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== david ===
Pré-carregando 462 frames...
  Done em 0.4s
Pasta: experimento_6
Pré-carregando 462 frames...
  Done em 0.3s


TrWisard1: 100%|██████████| 461/461 [00:36<00:00, 12.53it/s]


Pré-carregando 462 frames...
  Done em 0.4s


TrWisard2: 100%|██████████| 461/461 [00:31<00:00, 14.43it/s]


  TrWisard1     CLE=4.04 px  Jaccard=88.96%  FPS=12.6
  TrWisard2     CLE=6.21 px  Jaccard=83.78%  FPS=14.4


In [10]:
# ── faceocc ───────────────────────────────────────────────────────────────────
dataset = 'faceocc'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== faceocc ===
Pré-carregando 888 frames...
  Done em 1.0s
Pasta: experimento_6
Pré-carregando 888 frames...
  Done em 0.7s


TrWisard1: 100%|██████████| 887/887 [00:53<00:00, 16.68it/s]


Pré-carregando 888 frames...
  Done em 0.9s


TrWisard2: 100%|██████████| 887/887 [00:12<00:00, 70.24it/s]


  TrWisard1     CLE=5.28 px  Jaccard=90.17%  FPS=16.7
  TrWisard2     CLE=5.19 px  Jaccard=90.37%  FPS=70.1


In [11]:
# ── faceocc2 ──────────────────────────────────────────────────────────────────
dataset = 'faceocc2'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== faceocc2 ===
Pré-carregando 816 frames...
  Done em 0.9s
Pasta: experimento_6
Pré-carregando 816 frames...
  Done em 0.7s


TrWisard1: 100%|██████████| 815/815 [01:27<00:00,  9.37it/s]


Pré-carregando 816 frames...
  Done em 0.8s


TrWisard2: 100%|██████████| 815/815 [00:24<00:00, 33.81it/s]


  TrWisard1     CLE=11.84 px  Jaccard=75.29%  FPS=9.4
  TrWisard2     CLE=11.45 px  Jaccard=75.53%  FPS=33.8


In [12]:
# ── sylv ──────────────────────────────────────────────────────────────────────
dataset = 'sylv'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== sylv ===
Pré-carregando 1345 frames...
  Done em 0.8s
Pasta: experimento_5
Pré-carregando 1345 frames...
  Done em 0.7s


TrWisard1: 100%|██████████| 1344/1344 [00:09<00:00, 149.05it/s]


Pré-carregando 1345 frames...
  Done em 0.7s


TrWisard2: 100%|██████████| 1344/1344 [00:17<00:00, 77.26it/s]


  TrWisard1     CLE=10.28 px  Jaccard=64.43%  FPS=148.9
  TrWisard2     CLE=11.29 px  Jaccard=62.19%  FPS=77.1


In [13]:
# ── tiger1 ────────────────────────────────────────────────────────────────────
dataset = 'tiger1'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== tiger1 ===
Pré-carregando 354 frames...
  Done em 0.4s
Pasta: experimento_5
Pré-carregando 354 frames...
  Done em 0.3s


TrWisard1: 100%|██████████| 353/353 [00:15<00:00, 23.17it/s]


Pré-carregando 354 frames...
  Done em 0.3s


TrWisard2: 100%|██████████| 353/353 [00:46<00:00,  7.67it/s]


  TrWisard1     CLE=5.91 px  Jaccard=72.25%  FPS=23.2
  TrWisard2     CLE=7.39 px  Jaccard=70.01%  FPS=7.7


In [14]:
# ── tiger2 ────────────────────────────────────────────────────────────────────
dataset = 'tiger2'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

preds_1, elapsed_1 = run_isolated('tr_wisard1', dataset, params_1)
preds_2, elapsed_2 = run_isolated('tr_wisard2', dataset, params_2)

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== tiger2 ===
Pré-carregando 365 frames...
  Done em 0.4s
Pasta: experimento_5
Pré-carregando 365 frames...
  Done em 0.3s


TrWisard1: 100%|██████████| 364/364 [00:01<00:00, 206.68it/s]


Pré-carregando 365 frames...
  Done em 0.3s


TrWisard2: 100%|██████████| 364/364 [00:09<00:00, 39.96it/s]


  TrWisard1     CLE=8.50 px  Jaccard=62.85%  FPS=205.6
  TrWisard2     CLE=11.11 px  Jaccard=54.34%  FPS=39.9


## Tabela resumo
Execute após rodar os datasets desejados acima.

In [15]:
rows = []
for ds, r in summary.items():
    for mode, m in r.items():
        rows.append({
            'Dataset':           ds,
            'Modo':              mode,
            'CLE medio (px)':    round(m['mean_cle'], 2),
            'Jaccard medio (%)': round(m['mean_jaccard'], 2),
            'FPS':               round(m['fps'], 1),
        })

df = pd.DataFrame(rows)
df

,Dataset,Modo,CLE medio (px),Jaccard medio (%),FPS
0,dollar,Tr-WiSARD1,4.82,84.64,32.5
1,dollar,Tr-WiSARD2,2.58,92.09,41.5
2,david,TrWisard1,4.04,88.96,12.6
3,david,TrWisard2,6.21,83.78,14.4
4,faceocc,TrWisard1,5.28,90.17,16.7
5,faceocc,TrWisard2,5.19,90.37,70.1
6,faceocc2,TrWisard1,11.84,75.29,9.4
7,faceocc2,TrWisard2,11.45,75.53,33.8
8,sylv,TrWisard1,10.28,64.43,148.9
9,sylv,TrWisard2,11.29,62.19,77.1
